# Act III — The Family-Formation Squeeze
### *The Diverging American Dream: Can America Afford a Family?*

**Story beat:** Acts I and II showed *what* diverged and *who* it hit.
Act III asks the human question: what does this actually mean for the
decision to start a family? We follow the money — from the household
budget that makes the abstract personal, to the itemized cost of raising
a child, to the fertility rate that records the answer Americans are
giving with their lives.

| # | Visualization | Type | Requirement |
|---|---|---|---|
| 1 | Your 1960 vs. 2022 Household Budget | **Interactive calculator** | ✅ Interactive |
| 2 | The Cost of a Child — 1960 vs. 2023 | **Static infographic** | ✅ Infographic + Static |
| 3 | The Housing-Fertility Nexus | **Static dual-axis** | ✅ Static + Temporal |
| 4 | Scrub Through Six Decades | **Linked view** | ✅ Linked View |

---


## Setup — Imports & Theme

In [16]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from theme import (
    THEME, C,
    apply_theme, apply_dark_theme,
    annotate_threshold, add_recession_bands, watermark,
    PLOTLY_LAYOUT
)

RAW = "/Users/ishika/Desktop/MS/Scholarship_Project/Data/"   # adjust if your folder differs

print("✓ Imports complete")
print(f"  Red  (rising costs) : {C.red}")
print(f"  Blue (wages/income) : {C.blue}")
print(f"  Gold (reference)    : {C.gold}")


✓ Imports complete
  Red  (rising costs) : #B84A2E
  Blue (wages/income) : #2A6FAF
  Gold (reference)    : #C8872A


## Load & Prepare All Act III Data

All series loaded directly from raw FRED CSVs — same pattern as Acts I and II.

In [17]:
# ── FRED loader (identical to Acts I & II) ───────────────────
from unittest import result


def load_fred(filename, col_rename=None, start_year=1960, end_year=2025):
    df = pd.read_csv(RAW + filename)

    # Parse M/D/YY strings explicitly and fix two-digit-year rollover
    # e.g., "1/1/60" -> 1960 not 2060
    df['DATE'] = pd.to_datetime(df['DATE'], format='%m/%d/%y', errors='coerce')
    rollover = df['DATE'].dt.year > end_year
    if rollover.any():
        df.loc[rollover, 'DATE'] = df.loc[rollover, 'DATE'] - pd.DateOffset(years=100)

    df['year'] = df['DATE'].dt.year
    val_col = [c for c in df.columns
               if c not in ['DATE', 'year']][0]
    df = df.rename(columns={val_col: col_rename or val_col})
    name = col_rename or val_col
    df = df.dropna(subset=[name])
    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    #return df.groupby('year')[name].mean()
    result = df.groupby('year')[name].mean()
    result.index = result.index.astype(int)
    return result


def index_to_100(series, base_year):
    base = series.loc[base_year] if base_year in series.index \
           else series.dropna().iloc[0]
    return (series / base * 100).round(2)


# ── EPI date parser (same as Act II) ─────────────────────────
def parse_epi_year(d):
    yr = int(str(d).split('/')[-1])
    if yr < 100:
        return (2000 + yr) if yr <= 30 else (1900 + yr)
    return yr


# ── FRED series ───────────────────────────────────────────────
s_tfr      = load_fred('SPDYNTFRTINUSA.csv', 'tfr',                start_year=1960)
s_shelter  = load_fred('CUSR0000SAH1.csv',   'shelter_cpi')
s_tuition  = load_fred('CUUR0000SEEB.csv',   'tuition_cpi',        start_year=1978)
s_medical  = load_fred('CPIMEDSL.csv',       'medical_cpi')
s_hospital = load_fred('CUUR0000SEMC.csv',   'hospital_cpi',       start_year=1967)
s_apparel  = load_fred('CPIAPPSL.csv',       'apparel_cpi')
s_food     = load_fred('CPIFABSL.csv',       'food_cpi',           start_year=1967)
s_cpi_all  = load_fred('CPIAUCSL.csv',       'cpi_all')
s_wages    = load_fred('AHETPI.csv',         'avg_hourly_earnings',start_year=1964)
s_income   = load_fred('MEFAINUSA672N.csv',  'real_median_fam_income')
s_home     = load_fred('MSPUS.csv',          'median_home_price',  start_year=1963)
s_mortgage = load_fred('MORTGAGE30US.csv',   'mortgage_rate',      start_year=1971)
s_hpi      = load_fred('USSTHPI.csv',        'hpi',                start_year=1975)

# ── EPI wage percentiles ──────────────────────────────────────
wp_raw = pd.read_csv(RAW + 'epi_wage_percentiles.csv')
wp_raw['year'] = wp_raw['date'].apply(parse_epi_year)
wp_raw = wp_raw.drop(columns='date').set_index('year').sort_index()
wp_raw.columns = ['p10','p20','p30','p40','p50','p60','p70','p80','p90']

# ── Derived series ────────────────────────────────────────────
# Price-to-income: nominal home price / nominal family income
cpi_2023    = s_cpi_all.loc[2023]
mfi_nominal = s_income * (s_cpi_all / cpi_2023)
pti         = (s_home / mfi_nominal).round(2)

# Index series to 1967=100 for linked view
BASE_YEAR   = 1967
shelter_idx = index_to_100(s_shelter, BASE_YEAR)
wages_idx   = index_to_100(s_wages,   BASE_YEAR)

print("✓ All Act III series loaded")
for name, s in [
    ('TFR (SPDYNTFRTINUSA)',   s_tfr),
    ('Shelter CPI',            s_shelter),
    ('Tuition+Childcare CPI',  s_tuition),
    ('Avg Hourly Earnings',    s_wages),
    ('Real Median Fam Income', s_income),
    ('Median Home Price',      s_home),
    ('Mortgage Rate',          s_mortgage),
    ('Wage p10',               wp_raw['p10']),
]:
    print(f"  {name:28s}: {int(s.index.min())}–{int(s.index.max())}  ({len(s)} obs)")

print(f"\n  TFR: {s_tfr.loc[1960]:.3f} (1960) → {s_tfr.loc[2007]:.3f} (2007) → "
      f"{s_tfr.iloc[-1]:.3f} ({int(s_tfr.index.max())})")
print(f"  PTI: {pti.loc[1963]:.2f}x (1963)  →  {pti.loc[2022]:.2f}x (2022)")
print(f"  Shelter index 2024: {shelter_idx.loc[2024]:.0f}  (1967=100)")


✓ All Act III series loaded
  TFR (SPDYNTFRTINUSA)        : 1960–2024  (65 obs)
  Shelter CPI                 : 1960–2025  (66 obs)
  Tuition+Childcare CPI       : 1978–2025  (48 obs)
  Avg Hourly Earnings         : 1964–2025  (62 obs)
  Real Median Fam Income      : 1960–2024  (65 obs)
  Median Home Price           : 1963–2025  (63 obs)
  Mortgage Rate               : 1971–2025  (55 obs)
  Wage p10                    : 1973–2025  (53 obs)

  TFR: 3.654 (1960) → 2.120 (2007) → 1.627 (2024)
  PTI: 3.43x (1963)  →  4.56x (2022)
  Shelter index 2024: 1393  (1967=100)


---
## Visualization 1 — "Your 1960 vs. 2022 Household Budget"
### Interactive Calculator · Dropdown + Stacked Bar

Pick an income quintile from the dropdown. The two bars update to show
that household's budget in 1960 vs. 2022 — each category as a share of
after-tax income. The key story: *family-formation costs* (housing +
healthcare + education & childcare) take a dramatically larger bite today.

> **Sources:** BLS Consumer Expenditure Survey historical tables and
> USDA *Expenditures on Children by Families* (1960 & 2015 reports).
> Census Table H-3 for income levels, inflation-adjusted to 2023 dollars.


In [18]:
BUDGET = {
    'q1': {
        'label':       'Lowest 20%  (Bottom Quintile)',
        'income_1960': 17000, 'income_2022': 16000,
        1960: {'Housing':27,'Food':35,'Healthcare':5,
               'Transportation':13,'Education & Childcare':2,'Everything Else':18},
        2022: {'Housing':41,'Food':17,'Healthcare':9,
               'Transportation':17,'Education & Childcare':4,'Everything Else':12},
    },
    'q2': {
        'label':       '2nd Quintile',
        'income_1960': 34000, 'income_2022': 43000,
        1960: {'Housing':26,'Food':32,'Healthcare':5,
               'Transportation':14,'Education & Childcare':2,'Everything Else':21},
        2022: {'Housing':37,'Food':15,'Healthcare':9,
               'Transportation':18,'Education & Childcare':5,'Everything Else':16},
    },
    'q3': {
        'label':       'Middle 20%  (Middle Quintile)',
        'income_1960': 52000, 'income_2022': 72000,
        1960: {'Housing':25,'Food':28,'Healthcare':5,
               'Transportation':14,'Education & Childcare':3,'Everything Else':25},
        2022: {'Housing':33,'Food':13,'Healthcare':9,
               'Transportation':18,'Education & Childcare':7,'Everything Else':20},
    },
    'q4': {
        'label':       '4th Quintile',
        'income_1960': 72000, 'income_2022': 113000,
        1960: {'Housing':24,'Food':24,'Healthcare':5,
               'Transportation':15,'Education & Childcare':3,'Everything Else':29},
        2022: {'Housing':29,'Food':11,'Healthcare':8,
               'Transportation':17,'Education & Childcare':8,'Everything Else':27},
    },
    'q5': {
        'label':       'Highest 20%  (Top Quintile)',
        'income_1960': 130000, 'income_2022': 232000,
        1960: {'Housing':22,'Food':20,'Healthcare':4,
               'Transportation':14,'Education & Childcare':4,'Everything Else':36},
        2022: {'Housing':25,'Food': 9,'Healthcare':7,
               'Transportation':15,'Education & Childcare':9,'Everything Else':35},
    },
}

CATEGORIES = ['Housing','Healthcare','Education & Childcare',
              'Food','Transportation','Everything Else']
CAT_COLORS = {
    'Housing':               C.red,
    'Healthcare':            C.orange,
    'Education & Childcare': C.purple,
    'Food':                  C.green,
    'Transportation':        C.blue_light,
    'Everything Else':       C.gray_light,
}

QUINTILE_KEYS = ['q1','q2','q3','q4','q5']
N_CATS        = len(CATEGORIES)

fig_calc = go.Figure()

print(dir(C))

for qi, qk in enumerate(QUINTILE_KEYS):
    q = BUDGET[qk]
    inc60, inc22 = q['income_1960'], q['income_2022']
    s60, s22     = q[1960], q[2022]
    visible      = (qi == 2)   # default: middle quintile

    for cat in CATEGORIES:
        v60 = s60[cat];  d60 = round(inc60 * v60 / 100)
        v22 = s22[cat];  d22 = round(inc22 * v22 / 100)
        fig_calc.add_trace(go.Bar(
            name=cat,
            x=['1960', '2022'],
            y=[v60, v22],
            marker_color=CAT_COLORS[cat],
            marker_line=dict(width=0),
            text=[f'{v60}%<br>${d60:,}', f'{v22}%<br>${d22:,}'],
            textposition='inside',
            textfont=dict(size=11, color='white', family=THEME['font_mono']),
            customdata=[d60, d22],
            hovertemplate=(f'<b>{cat}</b><br>Year: %{{x}}<br>'
                           'Share: %{y}%<br>~$%{customdata:,}<extra></extra>'),
            visible=visible,
            showlegend=(qi == 2),
        ))

buttons = []
for qi, qk in enumerate(QUINTILE_KEYS):
    q   = BUDGET[qk]
    s60 = q[1960]; s22 = q[2022]
    fam60 = s60['Housing']+s60['Healthcare']+s60['Education & Childcare']
    fam22 = s22['Housing']+s22['Healthcare']+s22['Education & Childcare']
    vis = [False]*(len(QUINTILE_KEYS)*N_CATS)
    for ci in range(N_CATS): vis[qi*N_CATS+ci] = True
    buttons.append(dict(
        label=q['label'], method='update',
        args=[{'visible': vis},
              {'title': {'text': (f'<b>Your Household Budget: 1960 vs. 2022</b><br>'
                                  f'<sup>{q["label"]} · 2023 real dollars · '
                                  f'Housing+healthcare+education: {fam60}% → {fam22}%</sup>'),
                         'font': dict(size=THEME['font_size']['lg'],
                                      family=THEME['font'])}}],
    ))

q3  = BUDGET['q3']
f60 = q3[1960]['Housing']+q3[1960]['Healthcare']+q3[1960]['Education & Childcare']
f22 = q3[2022]['Housing']+q3[2022]['Healthcare']+q3[2022]['Education & Childcare']

apply_theme(fig_calc)
fig_calc.update_layout(
    barmode='stack',
    title=dict(
        text=(f'<b>Your Household Budget: 1960 vs. 2022</b><br>'
              f'<sup>Middle 20% · 2023 real dollars · '
              f'Housing+healthcare+education: {f60}% → {f22}%</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    updatemenus=[dict(
        buttons=buttons, direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.25, yanchor='top',
        bgcolor=THEME['background'],
        font=dict(size=12, family=THEME['font_sans']),
        pad=dict(r=10, t=5),
    )],
    annotations=[dict(
        x=0.0, y=1.13, xref='paper', yref='paper',
        text='← Select your income group:',
        showarrow=False,
        font=dict(size=11, color=C.muted, family=THEME['font_mono']),
    )],
    xaxis=dict(title='Year', tickfont=dict(size=14, family=THEME['font'])),
    yaxis=dict(title='Share of After-Tax Income (%)',
               range=[0,115], ticksuffix='%'),
    legend=dict(orientation='h', x=0, y=-0.18, xanchor='left',
                font=dict(size=11), tracegroupgap=0),
    bargap=0.35, height=580,
    margin=dict(t=160, b=130, l=70, r=40),
)
watermark(fig_calc, ('Sources: BLS Consumer Expenditure Survey · '
                     'USDA Expenditures on Children · Census Table H-3 · '
                     '2023 real dollars'))
fig_calc.show()

print("\n── Family-formation cost burden: housing + healthcare + education ──")
print(f"  {'Quintile':32s}  1960   2022   Change")
print("  " + "-"*55)
for qk in QUINTILE_KEYS:
    q = BUDGET[qk]
    f60 = q[1960]['Housing']+q[1960]['Healthcare']+q[1960]['Education & Childcare']
    f22 = q[2022]['Housing']+q[2022]['Healthcare']+q[2022]['Education & Childcare']
    print(f"  {q['label']:32s}  {f60:>3d}%   {f22:>3d}%  {f22-f60:>+4d} pp")


['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'bg', 'bg_dark', 'blue', 'blue_fill', 'blue_light', 'cpi', 'fertility', 'gold', 'gold_fill', 'gold_light', 'gray', 'gray_light', 'green', 'green_fill', 'green_light', 'grid', 'housing', 'map_seq', 'muted', 'orange', 'purple', 'red', 'red_fill', 'red_light', 'text', 'wages']



── Family-formation cost burden: housing + healthcare + education ──
  Quintile                          1960   2022   Change
  -------------------------------------------------------
  Lowest 20%  (Bottom Quintile)      34%    54%   +20 pp
  2nd Quintile                       33%    51%   +18 pp
  Middle 20%  (Middle Quintile)      33%    49%   +16 pp
  4th Quintile                       32%    45%   +13 pp
  Highest 20%  (Top Quintile)        30%    41%   +11 pp


---
## Visualization 2 — "The Cost of a Child: 1960 vs. 2023"
### Static Infographic — 5-Panel Subplot

Side-by-side: every major cost milestone from delivery through college,
all in 2023 real dollars. Blue = 1960, Red = 2023.

> **Sources**: AHRQ (hospital delivery) · DOL NDCP (childcare) ·
> College Board (tuition) · USDA Cost of Raising a Child reports ·
> All 1960 nominal values deflated to 2023$ via CPIAUCSL.


In [19]:
CHILD_COSTS = [
    {'milestone':'Hospital Delivery',      'cost_1960':2100,   'cost_2023':18865,
     'note_1960':'~$200 nominal',          'note_2023':'Avg. facility fee (AHRQ)'},
    {'milestone':'Infant Childcare (1 yr)','cost_1960':0,      'cost_2023':9100,
     'note_1960':'Single-earner norm',     'note_2023':'National median (DOL NDCP)'},
    {'milestone':'Public College (1 yr)',  'cost_1960':2400,   'cost_2023':11260,
     'note_1960':'~$243 nominal (1963)',   'note_2023':'In-state (College Board)'},
    {'milestone':'Total: Birth to Age 18', 'cost_1960':258000, 'cost_2023':310000,
     'note_1960':'USDA middle-income',     'note_2023':'USDA 2015 projected'},
    {'milestone':'% of Annual Income',     'cost_1960':15,     'cost_2023':27,
     'note_1960':'Middle quintile',        'note_2023':'Middle quintile',
     'is_pct': True},
]

positions = [(1,1),(1,2),(1,3),(2,1),(2,2)]
fig_info  = make_subplots(
    rows=2, cols=3,
    subplot_titles=[c['milestone'] for c in CHILD_COSTS],
    vertical_spacing=0.22, horizontal_spacing=0.10,
)

for (cost, (row, col)) in zip(CHILD_COSTS, positions):
    is_pct = cost.get('is_pct', False)
    v60, v23 = cost['cost_1960'], cost['cost_2023']
    fig_info.add_trace(go.Bar(
        x=['1960','2023'], y=[v60, v23],
        marker_color=[C.blue, C.red], marker_line=dict(width=0),
        text=([f'{v60}%',f'{v23}%'] if is_pct
              else [f'${v60:,}',f'${v23:,}']),
        textposition='outside',
        textfont=dict(size=13, family=THEME['font_mono'], color=THEME['text']),
        showlegend=False,
        hovertemplate=('%{x}: %{y}%<extra></extra>' if is_pct
                       else '%{x}: $%{y:,}<extra></extra>'),
    ), row=row, col=col)
    fig_info.update_yaxes(
        ticksuffix='%' if is_pct else '',
        tickformat='' if is_pct else '$,',
        range=[0, v23*1.35], row=row, col=col,
    )
    mult_text = f'<b>{v23/v60:.1f}×</b> more' if v60 > 0 else '<b>New cost</b>'
    idx = positions.index((row,col))
    xref = 'x domain' if idx == 0 else f'x{idx+1} domain'
    yref = 'y domain' if idx == 0 else f'y{idx+1} domain'
    fig_info.add_annotation(
        row=row, col=col, x=0.5, y=0.93, xref=xref, yref=yref,
        text=mult_text, showarrow=False,
        font=dict(size=11, color=C.red, family=THEME['font_mono']),
        bgcolor=THEME['background'], bordercolor=C.red, borderwidth=1,
    )

apply_theme(fig_info)
fig_info.update_layout(
    title=dict(
        text=('<b>The Cost of a Child: 1960 vs. 2023</b><br>'
              '<sup>All values in 2023 real dollars · Blue = 1960 · Red = 2023</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    height=640, showlegend=False,
    margin=dict(t=110, b=80, l=60, r=40),
)
watermark(fig_info, ('Sources: AHRQ · DOL NDCP · College Board · '
                     'USDA Expenditures on Children · CPI via FRED'))
fig_info.show()

print("\n── Cost of a Child (2023 real $) ────────────────────────────────")
for c in CHILD_COSTS:
    v60, v23 = c['cost_1960'], c['cost_2023']
    mult = f'{v23/v60:.1f}x' if v60 > 0 else 'new cost'
    if c.get('is_pct'):
        print(f"  {c['milestone']:28s}: {v60}% → {v23}%")
    else:
        print(f"  {c['milestone']:28s}: ${v60:>8,} → ${v23:>8,}  ({mult})")



── Cost of a Child (2023 real $) ────────────────────────────────
  Hospital Delivery           : $   2,100 → $  18,865  (9.0x)
  Infant Childcare (1 yr)     : $       0 → $   9,100  (new cost)
  Public College (1 yr)       : $   2,400 → $  11,260  (4.7x)
  Total: Birth to Age 18      : $ 258,000 → $ 310,000  (1.2x)
  % of Annual Income          : 15% → 27%


---
## Visualization 3 — "The Housing-Fertility Nexus"
### Static Dual-Axis Annotated Chart

**Left axis**: Total Fertility Rate · **Right axis**: Shelter CPI (1967=100).

Every housing surge — oil shock, dot-com run-up, post-2008 recovery,
COVID — coincides with a fertility drop or stall. Couillard (2024)
estimates ~51% of 2000s→2010s fertility decline is attributable to
rising housing costs.


In [23]:
fig_fert = make_subplots(specs=[[{'secondary_y': True}]])
REPLACEMENT = 2.1

common_yrs   = sorted(set(s_tfr.dropna().index) & set(shelter_idx.dropna().index))
tfr_vals     = [float(s_tfr.loc[y])      for y in common_yrs]
shelter_vals = [float(shelter_idx.loc[y]) for y in common_yrs]

# Red fill below replacement
below_y = [v if v <= REPLACEMENT else REPLACEMENT for v in tfr_vals]
fig_fert.add_trace(go.Scatter(
    x=common_yrs+common_yrs[::-1],
    y=below_y+[REPLACEMENT]*len(common_yrs),
    fill='toself', fillcolor=C.red_fill,
    line=dict(width=0), showlegend=False, hoverinfo='skip',
), secondary_y=False)

fig_fert.add_trace(go.Scatter(
    x=common_yrs, y=tfr_vals, name='Total Fertility Rate (left)',
    mode='lines', line=dict(color=C.gold, width=3.2),
    hovertemplate='<b>TFR</b> · %{x}<br>%{y:.3f}<extra></extra>',
), secondary_y=False)

fig_fert.add_trace(go.Scatter(
    x=common_yrs, y=shelter_vals, name='Shelter CPI (right, 1967=100)',
    mode='lines', line=dict(color=C.red, width=2.5, dash='dot'),
    hovertemplate='<b>Shelter CPI</b> · %{x}<br>Index: %{y:.0f}<extra></extra>',
), secondary_y=True)

fig_fert.add_hline(
    y=REPLACEMENT, line_dash='dash', line_color=C.red, line_width=1.5,
    annotation_text='Replacement (2.1)', annotation_position='top left',
    annotation_font=dict(size=10, color=C.red, family=THEME['font_mono']),
    secondary_y=False,
)

for x0, x1, lbl in [(1973,1980,'Oil shock'),(1995,2006,'Housing bubble'),
                     (2012,2019,'Post-GR'),(2020,2024,'COVID surge')]:
    fig_fert.add_vrect(x0=x0, x1=x1, fillcolor='rgba(184,74,46,0.06)',
                       line_width=0, annotation_text=lbl,
                       annotation_position='top left',
                       annotation_font=dict(size=9, color=C.red,
                                            family=THEME['font_mono']))

for yr, txt, ax, ay in [
    (1960, f'Baby Boom peak\n{s_tfr.loc[1960]:.2f}',       -15, -45),
    (2007, f'Last peak {s_tfr.loc[2007]:.2f}',               15, -40),
    (int(s_tfr.index.max()),
     f'Record low\n{s_tfr.iloc[-1]:.2f}',                   15, -35),
]:
    if yr in s_tfr.index:
        fig_fert.add_annotation(
            x=yr, y=float(s_tfr.loc[yr]), text=txt,
            showarrow=True, arrowhead=2, arrowcolor=C.gold,
            font=dict(size=10, color=C.text, family=THEME['font_mono']),
            bgcolor=THEME['background'], bordercolor=C.gold, borderwidth=1,
            ax=ax, ay=ay, secondary_y=False,
        )

fig_fert.add_annotation(
    x=2010, y=1.50,
    text=('<b>Couillard (2024)</b>:<br>Housing costs explain<br>'
          '~51% of fertility decline<br>2000s → 2010s'),
    showarrow=True, arrowhead=2, arrowcolor=C.muted, arrowwidth=1.5,
    font=dict(size=10, color=C.text, family=THEME['font']),
    bgcolor=THEME['background'], bordercolor=C.muted, borderwidth=1,
    ax=0, ay=65, secondary_y=False,
)

add_recession_bands(fig_fert)
apply_theme(fig_fert)

fig_fert.update_layout(
    title=dict(
        text=('<b>The Housing-Fertility Nexus, 1960–2024</b><br>'
              '<sup>Gold: Total Fertility Rate (left) · '
              'Red dashed: Shelter CPI 1967=100 (right)</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    xaxis=dict(title='Year', tickmode='linear', dtick=5, range=[1960,2025]),
    legend=dict(x=0.01, y=0.20, xanchor='left',
                bgcolor='rgba(245,240,232,0.9)'),
    height=520, margin=dict(t=100, b=70, l=70, r=90),
)
fig_fert.update_yaxes(title_text='Total Fertility Rate', range=[1.2,4.2],
                      tickformat='.2f', secondary_y=False)
fig_fert.update_yaxes(title_text='Shelter CPI (1967=100)', range=[0,1700],
                      secondary_y=True)

fig_fert.show()

corr = np.corrcoef(tfr_vals, shelter_vals)[0,1]
last_yr = int(s_tfr.index.max())
print(f"\n── Pearson r (TFR vs Shelter CPI, 1960-{last_yr}): {corr:.3f}")
for yr in [1960,1972,1976,2007,2020,last_yr]:
    if yr in s_tfr.index:
        flag = ' ← below replacement' if s_tfr[yr] < 2.1 else ''
        print(f"  {yr}: {s_tfr[yr]:.3f}{flag}")



── Pearson r (TFR vs Shelter CPI, 1960-2024): -0.555
  1960: 3.654
  1972: 2.010 ← below replacement
  1976: 1.738 ← below replacement
  2007: 2.120
  2020: 1.641 ← below replacement
  2024: 1.627 ← below replacement


---
## Visualization 4 — "Scrub Through Six Decades"
### Interactive Linked View — Three Coordinated Panels

A year slider controls all three panels simultaneously:
- **Top** — Shelter cost index vs. wages index (1967=100)
- **Middle** — Real hourly wage: P10 vs P90
- **Bottom** — Total fertility rate vs. 2.1 replacement

Press ▶ **Play** for an animated walkthrough of 1967–2024.


In [21]:
# Aligned years where all three series exist
lv_years = sorted(
    set(shelter_idx.dropna().index)
    & set(wages_idx.dropna().index)
    & set(s_tfr.dropna().index)
)
lv_years    = [y for y in lv_years if 1967 <= y <= 2024]
lv_shelter  = [float(shelter_idx.loc[y]) for y in lv_years]
lv_wages    = [float(wages_idx.loc[y])   for y in lv_years]
lv_tfr      = [float(s_tfr.loc[y])       for y in lv_years]
lv_p10      = [float(wp_raw.loc[y,'p10']) if y in wp_raw.index else None for y in lv_years]
lv_p90      = [float(wp_raw.loc[y,'p90']) if y in wp_raw.index else None for y in lv_years]

REPLACEMENT = 2.1

fig_lv = make_subplots(
    rows=3, cols=1, shared_xaxes=False,
    subplot_titles=(
        'Shelter Cost vs. Average Wages  (1967 = 100)',
        'Real Hourly Wage: 10th vs. 90th Percentile  (2023 $)',
        'Total Fertility Rate',
    ),
    vertical_spacing=0.09, row_heights=[0.34,0.33,0.33],
)

# ── Panel 1 ───────────────────────────────────────────────────
fig_lv.add_trace(go.Scatter(x=lv_years, y=lv_shelter, name='Shelter CPI (1967=100)',
    mode='lines', line=dict(color=C.red, width=2.5),
    hovertemplate='Shelter · %{x}: %{y:.0f}<extra></extra>'), row=1, col=1)
fig_lv.add_trace(go.Scatter(x=lv_years, y=lv_wages, name='Avg Hourly Earnings (1967=100)',
    mode='lines', line=dict(color=C.blue, width=2.5, dash='dot'),
    hovertemplate='Wages · %{x}: %{y:.0f}<extra></extra>'), row=1, col=1)
fig_lv.add_trace(go.Scatter(x=[lv_years[0]], y=[lv_shelter[0]], mode='markers',
    marker=dict(color=C.red, size=11, line=dict(color='white',width=2)),
    showlegend=False, hoverinfo='skip', name='_dot_shelter'), row=1, col=1)
fig_lv.add_trace(go.Scatter(x=[lv_years[0]], y=[lv_wages[0]], mode='markers',
    marker=dict(color=C.blue, size=11, line=dict(color='white',width=2)),
    showlegend=False, hoverinfo='skip', name='_dot_wages'), row=1, col=1)

# ── Panel 2 ───────────────────────────────────────────────────
p10c = [(y,v) for y,v in zip(lv_years,lv_p10) if v is not None]
p90c = [(y,v) for y,v in zip(lv_years,lv_p90) if v is not None]
fig_lv.add_trace(go.Scatter(x=[r[0] for r in p10c], y=[r[1] for r in p10c],
    name='10th Percentile Wage', mode='lines', line=dict(color=C.blue, width=2.5),
    hovertemplate='P10 · %{x}: $%{y:.2f}/hr<extra></extra>'), row=2, col=1)
fig_lv.add_trace(go.Scatter(x=[r[0] for r in p90c], y=[r[1] for r in p90c],
    name='90th Percentile Wage', mode='lines', line=dict(color=C.red, width=2.5),
    hovertemplate='P90 · %{x}: $%{y:.2f}/hr<extra></extra>'), row=2, col=1)
p10s = next((v for y,v in p10c if y==lv_years[0]), None)
p90s = next((v for y,v in p90c if y==lv_years[0]), None)
fig_lv.add_trace(go.Scatter(x=[lv_years[0]] if p10s else [], y=[p10s] if p10s else [],
    mode='markers', marker=dict(color=C.blue, size=11, line=dict(color='white',width=2)),
    showlegend=False, hoverinfo='skip', name='_dot_p10'), row=2, col=1)
fig_lv.add_trace(go.Scatter(x=[lv_years[0]] if p90s else [], y=[p90s] if p90s else [],
    mode='markers', marker=dict(color=C.red, size=11, line=dict(color='white',width=2)),
    showlegend=False, hoverinfo='skip', name='_dot_p90'), row=2, col=1)

# ── Panel 3 ───────────────────────────────────────────────────
below_lv = [v if v<=REPLACEMENT else REPLACEMENT for v in lv_tfr]
fig_lv.add_trace(go.Scatter(x=lv_years+lv_years[::-1],
    y=below_lv+[REPLACEMENT]*len(lv_years), fill='toself',
    fillcolor=C.red_fill, line=dict(width=0),
    showlegend=False, hoverinfo='skip'), row=3, col=1)
fig_lv.add_trace(go.Scatter(x=lv_years, y=lv_tfr, name='Total Fertility Rate',
    mode='lines', line=dict(color=C.gold, width=2.8),
    hovertemplate='TFR · %{x}: %{y:.3f}<extra></extra>'), row=3, col=1)
fig_lv.add_hline(y=REPLACEMENT, row=3, col=1, line_dash='dash',
    line_color=C.red, line_width=1.5, annotation_text='Replacement (2.1)',
    annotation_position='top right',
    annotation_font=dict(size=9, color=C.red, family=THEME['font_mono']))
fig_lv.add_trace(go.Scatter(x=[lv_years[0]], y=[lv_tfr[0]], mode='markers',
    marker=dict(color=C.gold, size=11, line=dict(color='white',width=2)),
    showlegend=False, hoverinfo='skip', name='_dot_tfr'), row=3, col=1)

total_traces = len(fig_lv.data)
# Dot traces: shelter_dot=2, wages_dot=3, p10_dot=6, p90_dot=7, tfr_dot=last
DOT_IDX = [2, 3, 6, 7, total_traces-1]

# ── Frames ────────────────────────────────────────────────────
frames = []
for i, yr in enumerate(lv_years):
    p10v = lv_p10[i]; p90v = lv_p90[i]
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[yr], y=[lv_shelter[i]]),
            go.Scatter(x=[yr], y=[lv_wages[i]]),
            go.Scatter(x=[yr] if p10v else [], y=[p10v] if p10v else []),
            go.Scatter(x=[yr] if p90v else [], y=[p90v] if p90v else []),
            go.Scatter(x=[yr], y=[lv_tfr[i]]),
        ],
        traces=DOT_IDX, name=str(yr),
    ))
fig_lv.frames = frames

sliders = [dict(
    active=0,
    currentvalue=dict(prefix='Year: ',
                      font=dict(size=14, color=C.text, family=THEME['font_mono']),
                      xanchor='center'),
    pad=dict(b=10, t=10), len=0.92, x=0.04,
    steps=[dict(
        args=[[str(yr)], dict(frame=dict(duration=80, redraw=True),
                               mode='immediate', transition=dict(duration=40))],
        label=str(yr) if yr%10==0 else '', method='animate',
    ) for yr in lv_years],
)]

play_pause = [dict(
    type='buttons', showactive=False, y=0, x=0.01, xanchor='left',
    yanchor='top', pad=dict(t=60, r=10),
    buttons=[
        dict(label='▶ Play', method='animate',
             args=[None, dict(frame=dict(duration=120, redraw=True),
                              fromcurrent=True, transition=dict(duration=60))]),
        dict(label='⏸ Pause', method='animate',
             args=[[None], dict(frame=dict(duration=0, redraw=False),
                                mode='immediate', transition=dict(duration=0))]),
    ],
)]

add_recession_bands(fig_lv, row=1, col=1)
add_recession_bands(fig_lv, row=2, col=1)
add_recession_bands(fig_lv, row=3, col=1)
apply_theme(fig_lv)

fig_lv.update_layout(
    title=dict(
        text=('<b>Scrub Through Six Decades</b><br>'
              '<sup>Drag the slider or press ▶ Play · '
              'All three panels update together</sup>'),
        font=dict(size=THEME['font_size']['lg'], family=THEME['font']),
    ),
    sliders=sliders, updatemenus=play_pause,
    height=740, margin=dict(t=100, b=130, l=75, r=40),
    legend=dict(orientation='h', x=0, y=-0.22, font=dict(size=10)),
)
fig_lv.update_yaxes(title_text='Index (1967=100)',   row=1, col=1)
fig_lv.update_yaxes(title_text='$/hr (2023 real $)', row=2, col=1, tickformat='$,.0f')
fig_lv.update_yaxes(title_text='Births per woman',   row=3, col=1,
                    tickformat='.2f', range=[1.2,4.0])
fig_lv.update_xaxes(title_text='Year', tickmode='linear', dtick=10, row=3, col=1)

watermark(fig_lv, ('Sources: BLS/FRED (CUSR0000SAH1, AHETPI) · '
                   'EPI (epi_wage_percentiles.csv) · '
                   'World Bank/CDC via FRED (SPDYNTFRTINUSA)'))
fig_lv.show()

print(f"\n✓ Linked view: {lv_years[0]}–{lv_years[-1]}, {len(frames)} frames, "
      f"{total_traces} traces")



✓ Linked view: 1967–2024, 58 frames, 11 traces


---
## Act III — Summary of Key Numbers

In [ ]:
# print("="*65)
# print("ACT III — KEY NUMBERS FOR WEBSITE CALLOUT CARDS")
# print("="*65)

# q3  = BUDGET['q3']
# f60 = q3[1960]['Housing']+q3[1960]['Healthcare']+q3[1960]['Education & Childcare']
# f22 = q3[2022]['Housing']+q3[2022]['Healthcare']+q3[2022]['Education & Childcare']
# print(f"\n── Budget (middle quintile, housing+healthcare+education) ──────")
# print(f"  1960: {f60}%  →  2022: {f22}%  (+{f22-f60} pp)")

# print("\n── Cost of a child (2023 real $) ────────────────────────────")
# for c in CHILD_COSTS:
#     v60, v23 = c['cost_1960'], c['cost_2023']
#     mult = f'{v23/v60:.1f}x' if v60>0 else 'new cost'
#     if c.get('is_pct'):
#         print(f"  {c['milestone']:28s}: {v60}% → {v23}%")
#     else:
#         print(f"  {c['milestone']:28s}: ${v60:>8,} → ${v23:>8,}  ({mult})")

# last_yr = int(s_tfr.index.max())
# print("\n── Fertility ─────────────────────────────────────────────────")
# print(f"  1960: {s_tfr.loc[1960]:.3f}  2007: {s_tfr.loc[2007]:.3f}  "
#       f"{last_yr}: {s_tfr.iloc[-1]:.3f}")
# print(f"  Decline 2007→{last_yr}: {((s_tfr.iloc[-1]/s_tfr.loc[2007])-1)*100:.1f}%")
# print(f"  Housing explains ~51% of 2000s-2010s decline (Couillard 2024)")
# print("\n"+"="*65)


ACT III — KEY NUMBERS FOR WEBSITE CALLOUT CARDS

── Budget (middle quintile, housing+healthcare+education) ──────
  1960: 33%  →  2022: 49%  (+16 pp)

── Cost of a child (2023 real $) ────────────────────────────


NameError: name 'CHILD_COSTS' is not defined

---
## Export Figures

In [22]:
import os
OUT_DIR = '../outputs/act3/'
os.makedirs(OUT_DIR, exist_ok=True)

figures = {
    'act3_fig1_budget_calculator': fig_calc,
    'act3_fig2_cost_of_child':     fig_info,
    'act3_fig3_fertility_housing': fig_fert,
    'act3_fig4_linked_view':       fig_lv,
}
for name, f in figures.items():
    f.write_html(OUT_DIR+name+'.html', include_plotlyjs='cdn', full_html=False,
                 config={'displayModeBar':True,'displaylogo':False,
                         'modeBarButtonsToRemove':['lasso2d','select2d']})
    print(f"  ✓ HTML: {OUT_DIR+name}.html")
    try:
        f.write_image(OUT_DIR+name+'.png', width=1200,
                      height=f.layout.height or 520, scale=2)
        print(f"  ✓ PNG : {OUT_DIR+name}.png")
    except Exception as e:
        print(f"  ⚠ PNG skipped — {e}")

print(f"\n✅  Act III export complete → {OUT_DIR}")
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"   {fn:48s}  {os.path.getsize(OUT_DIR+fn)/1024:6.1f} KB")


  ✓ HTML: ../outputs/act3/act3_fig1_budget_calculator.html
  ✓ PNG : ../outputs/act3/act3_fig1_budget_calculator.png
  ✓ HTML: ../outputs/act3/act3_fig2_cost_of_child.html
  ✓ PNG : ../outputs/act3/act3_fig2_cost_of_child.png
  ✓ HTML: ../outputs/act3/act3_fig3_fertility_housing.html
  ✓ PNG : ../outputs/act3/act3_fig3_fertility_housing.png
  ✓ HTML: ../outputs/act3/act3_fig4_linked_view.html
  ✓ PNG : ../outputs/act3/act3_fig4_linked_view.png

✅  Act III export complete → ../outputs/act3/
   act3_fig1_budget_calculator.html                    28.2 KB
   act3_fig1_budget_calculator.png                    191.5 KB
   act3_fig2_cost_of_child.html                        14.0 KB
   act3_fig2_cost_of_child.png                        246.5 KB
   act3_fig3_fertility_housing.html                    17.1 KB
   act3_fig3_fertility_housing.png                    282.1 KB
   act3_fig4_linked_view.html                          45.7 KB
   act3_fig4_linked_view.png                          371.9 KB
